# Spatial Statistics: Quantifying Spatial Organization Gene by Gene

The spatial scatter plots in `01_core_workflow.ipynb` suggest that cluster cells co-localize physically. Visual inspection is not a quantitative test. This notebook demonstrates spatial statistics functions in `astrolabe.tl` to the same question rigorously: is a given spatial pattern distinguishable from randomness, and if so, where, at what scale, and does it co-vary with a second signal?

Tests proceed from a single global statistic for a single gene up to a genome-wide scan. Every test requires a spatial graph from `tl.spatial_neighbors`.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import astrolabe as ab

## 0. Setup

This notebook is self-contained: the minimal pipeline from `01_core_workflow.ipynb` — load, QC-filter, normalize, cluster — runs in one block below. `tl.transcriptional_neighbors` builds the transcriptomic graph (`uns['neighbors']`) for clustering, and `tl.spatial_neighbors` builds the spatial graph (`uns['spatial_neighbors']`) the rest of this notebook needs — the two coexist independently, so the order they're built in doesn't matter.

Two genes anchor this notebook: `EPCAM` (epithelial cell adhesion molecule, a canonical epithelial/tumor marker enriched in this Lung Cancer FFPE sample) as the primary gene for the single-gene sections, and `PTPRC` (CD45, a pan-immune marker) as a second signal for the bivariate section. Both show clear spatial autocorrelation in this dataset, representing distinct biological compartments (tumor epithelium vs. immune infiltrate).

In [ ]:
mdata = ab.datasets.xenium()

ab.pipeline.filter_xenium(mdata)
ab.pp.normalize_total(mdata)
ab.pp.log1p(mdata)

ab.pp.highly_variable_genes(mdata, n_top_genes=2000)  # required before PCA/KNN on 154k cells
ab.tl.pca(mdata)
ab.tl.transcriptional_neighbors(mdata)
ab.tl.cluster(mdata, resolution=0.5, neighbors_key='neighbors')

gene = 'EPCAM'
gene2 = 'PTPRC'
assert gene in mdata['xenium'].var_names and gene2 in mdata['xenium'].var_names

# Spatial graph the rest of this notebook needs -- coexists independently
# of the transcriptomic graph built above.
ab.tl.spatial_neighbors(mdata, coord_type='delaunay', delaunay_max_dist=200)

ab.pl.image(mdata, image_key='morphology', channel=0).show()

mdata['xenium'].obs['leiden'].value_counts()

In [ ]:
mdata

## 1. Global autocorrelation (Moran's I)

Global Moran's I tests whether, across the whole tissue, nearby cells tend to have similar `EPCAM` expression rather than expression scattered independent of position. It is the spatial analogue of a correlation coefficient: each cell's standardized value is regressed against the average of its neighbors'. `I` near 0 means no spatial structure; positive `I` means clustering (similar values nearby); negative `I` means dispersion (neighbors tend to differ).

`tl.moran` reads the spatial graph via `uns['spatial_neighbors']` (the default `neighbors_key`, written above by `tl.spatial_neighbors`) and reports both an analytic z-score and a permutation-based empirical p-value.

In [ ]:
moran_result = ab.tl.moran(mdata, [gene, gene2])
moran_result

EPCAM's Moran's I ~ 0.5, z = 323, p_sim <= 0.001: one of the strongest autocorrelation signals in this dataset. The 2-disconnected-components warning from libpysal is innocuous: it means the spatial graph has isolated "islands" as evident by there being a small piece of disconnected tissue near the top of the section. PTPRC (I = 0.258, z = 174) is also strongly autocorrelated, though less so than EPCAM, consistent with immune infiltrate being spatially organized but spread more broadly than the tumor epithelial nests.

Moran's I relies on a particular formalization of "similar to neighbors" (a cross-product between deviations from the mean). Geary's C asks the same question with a different formalization: squared *differences* between neighbors rather than cross-products of deviations from the global mean, making it more sensitive to local dissimilarity than to global structure. Agreement between the two statistics is a sturdier conclusion than either alone. By convention `C < 1` indicates positive autocorrelation (matching positive Moran's `I`); `C > 1` indicates negative autocorrelation.

In [ ]:
geary_result = ab.tl.geary_c(mdata, [gene, gene2])
geary_result

Geary's C agrees: EPCAM C = 0.520, PTPRC C = 0.739 (both well below 1.0, both p_sim = 0.001), confirming clustering under both the cross-product formulation (Moran's I) and the difference formulation (Geary's C) — the conclusion does not depend on which spatial autocorrelation statistic is used.

## 2. Local indicators of spatial association (LISA)

A single global `I` can hide a lot: a gene could have a strong hotspot in one corner of the tissue and be randomly scattered everywhere else, and the global statistic would average that into a moderate, unremarkable number. Local indicators identify where the pattern occurs, not just whether it exists.

`tl.moran_local` (LISA) classifies every cell into one of four quadrants relative to its neighbors: **HH** (high surrounded by high, a hotspot), **LL** (low surrounded by low, a coldspot), and the spatial-outlier classes **HL**/**LH** (high cell in a low neighborhood, or vice versa). It returns a per-cell, per-key result.

In [ ]:
lisa = ab.tl.moran_local(mdata, [gene, gene2])

# moran_local returns a (key, obs_name) MultiIndex DataFrame -- pl.points colors
# by a var_name or obs column, so pull out one gene's quadrant labels into obs.
# Non-significant cells (p_adj >= 0.05) are labeled 'NS' rather than plotted
# with a quadrant call the test didn't actually support.
lisa_gene = lisa.loc[gene]
sig_quadrant = lisa_gene['quadrant'].astype(str).where(lisa_gene['p_adj'] < 0.05, 'NS')
mdata['xenium'].obs[f'{gene}_quadrant'] = sig_quadrant.to_numpy()
sig_quadrant.value_counts()

At p_adj < 0.05: LL (coldspot) >> HH (hotspot). LL cells are EPCAM-negative cells surrounded by EPCAM-negative neighbors — the stromal, immune, and endothelial regions that make up the majority of the tissue area. The smaller HH count reflects that the tumor epithelial nests, while highly clustered, cover less total tissue area than the stroma. HL outliers are potential invasion-front cells.

In [ ]:
(ab.pl.points(mdata, color=gene) | ab.pl.points(mdata, color=f'{gene}_quadrant')).show()

Side by side: raw expression on the left, LISA quadrant classification on the right. If `EPCAM` is genuinely spatially organized, the HH cells in the right panel should sit on top of the brightest patches on the left.

`tl.getis_ord` (local G*) answers a narrower version of the same question: it collapses LISA's four classes down to one axis (hot, cold, or not significant) and skips distinguishing spatial outliers. This is easier to read at a glance when the goal is simply locating where expression concentrates, without needing to distinguish an HH-cluster member from an HL-outlier.

In [ ]:
go = ab.tl.getis_ord(mdata, [gene])
mdata['xenium'].obs[f'{gene}_hotspot'] = go.loc[gene, 'type'].to_numpy()

ab.pl.points(mdata, color=f'{gene}_hotspot').show()

## 3. Bivariate spatial cross-correlation

So far every test has asked whether one signal correlates with *itself* across space. A different and equally common question is whether an epithelial marker (`EPCAM`) and a pan-immune marker (`PTPRC`/CD45) occupy the same or complementary regions. That is a spatial cross-correlation, not a single-gene autocorrelation: `tl.moran_bv` is the global version (one number for the whole tissue), and `tl.moran_local_bv` is the per-cell version. A negative or near-zero bivariate `I` here would be the biologically expected result, since tumor epithelium and immune infiltrate tend to occupy distinct tissue compartments — this doubles as a sanity check that the statistic behaves sensibly on a real biological contrast.

In [ ]:
bv = ab.tl.moran_bv(mdata, gene, gene2)
bv

Bivariate Moran's I = −0.260, p_sim = 0.001. The negative cross-correlation confirms that cells with high EPCAM tend to be surrounded by cells with low PTPRC, and vice versa: tumor epithelium and immune infiltrate occupy spatially complementary, not overlapping, tissue regions. A value of −0.260 represents ~26% of maximum possible anti-correlation between these two genes in space, which is biologically substantial for a tissue where there is no anatomical barrier between tumor and immune cells — only the spatial preferences driven by cell-cell adhesion and chemokine gradients.

In [ ]:
bv_local = ab.tl.moran_local_bv(mdata, gene, gene2)
pair_label = f'{gene}:{gene2}'
bv_pair = bv_local.loc[pair_label]

# Only assign a quadrant label where it's statistically significant (p_adj < 0.05);
# non-significant cells get 'NS' rather than a colocalization call the test
# didn't actually support.
sig_bv_quadrant = bv_pair['quadrant'].astype(str).where(bv_pair['p_adj'] < 0.05, 'NS')
mdata['xenium'].obs['bv_quadrant'] = sig_bv_quadrant.to_numpy()
print(sig_bv_quadrant.value_counts())

ab.pl.points(mdata, color='bv_quadrant').show()

`bv_quadrant` reuses the same HH/LL/HL/LH labels as the univariate LISA in section 2, but the pairing is asymmetric. Here, each label compares a cell's own `EPCAM` value against the *spatial lag of PTPRC*. Concretely, here **HH** = EPCAM-high cell in a PTPRC-high neighborhood, **LL** = EPCAM-low cell in a PTPRC-low neighborhood, and the outlier classes flip meaning relative to the univariate case: **HL** = EPCAM-high cell in a PTPRC-low neighborhood, **LH** = EPCAM-low cell in a PTPRC-high neighborhood. As above, cells that don't clear `p_adj < 0.05` are labeled `NS` rather than given a quadrant call the test doesn't actually support at that cell.

## 4. Smoothing: making a noisy local pattern visible

Single-cell expression of any one gene is noisy: dropout and segmentation error mean a single cell's count is a weak signal even where the underlying biology is genuinely spatially patterned. `tl.spatial_lag` replaces each cell's value with the (row-standardized) average of its neighbors', the same smoothing operation implicit in Moran's I, but kept as a per-cell value rather than collapsed into a single statistic so it can be visualized directly. Compare the raw signal to its spatial lag below.

In [ ]:
ab.tl.spatial_lag(mdata, gene)

(ab.pl.points(mdata, color=gene) | ab.pl.points(mdata, color=f'{gene}_lag')).show()

The spatial lag plot (EPCAM_lag) shows sharper compartment boundaries than the raw EPCAM expression: cells at the tumor-stroma boundary have intermediate values in the raw plot (half-expressing, half-dropout), but their spatial lag value is dominated by the majority expression in their neighborhood, resolving them clearly to one compartment. This also reduces zero-inflation: cells with zero raw EPCAM counts but inside a high-EPCAM neighborhood get a non-zero lag value, reducing the artificial zeros that would otherwise inflate a coldspot map.

## 5. Cluster-boundary robustness

The core workflow notebook clustered cells on expression (`leiden`) and noted that spatial coherence in a tissue plot was *suggestive* of a real anatomical correlate, but not quantified. `tl.spatial_silhouette` is a direct, per-cell way to probe that: it scores how well each cell separates from its *spatial* neighbors in expression space (PCA by default), signed and centered at 0 like a z-score. A cell with no same-cluster spatial neighbor at all, or with no cross-cluster neighbor at all (i.e. not on any boundary), scores exactly 0 -- 0 is not "ambiguous," it's the majority of a well-clustered tissue. Among the remaining boundary cells, positive means the cell fits its own cluster better than the neighboring one it borders; negative means the opposite, a likely-noisy assignment. This differs from a standard silhouette score, which compares against the nearest cluster in expression space regardless of physical location: here, sign and magnitude specifically describe *spatial* boundary quality, not expression-space outlierness.

In [ ]:
ab.tl.spatial_silhouette(mdata, groupby='leiden')

# spatial_silhouette is signed and centered at 0 (like a z-score) -- a
# diverging colormap centered at vcenter=0 is what makes that structure
# visible; the default sequential colormap washes real-but-modest
# variation (typically within roughly +/-0.5 here, not the full +/-1 range)
# into indistinguishable pale shades.
(ab.pl.points(mdata, color='leiden') | ab.pl.points(mdata, color='spatial_silhouette', cmap='astrolabe_div', vcenter=0.0, vmin=-1, vmax=1)).show()

Spatial silhouette scores range from −1 (a cell's nearest cluster in expression space is also its nearest cluster in physical space: it sits at a spatial boundary) to +1 (pure spatial-expression concordance). High average scores for a cluster mean that Leiden boundary is also a *spatial* boundary: these are the most anatomically distinct cell types. Low scores flag clusters that are transcriptomically distinct but physically interleaved with other types — in this dataset, most likely the immune clusters (2, 4, 8) that scatter through the stroma rather than forming compact nests.

## 6. Point-pattern clustering by cell type

Everything so far has tested a *continuous* signal (gene expression) against space. A different question applies to a *categorical* label like `leiden`: treating cells of one type as a binary point pattern (present/absent), whether they clump together at some characteristic spatial scale more than expected under complete spatial randomness. This doesn't need the spatial graph at all: Ripley's L works directly off cell centroids and a CSR (complete spatial randomness) simulation envelope, not a fixed neighbor graph, so it can probe *any* distance scale, not just the one baked into `spatial_neighbors`.

`tl.ripley_l(mdata, groupby='leiden')` runs this once per cluster and returns one DataFrame per group; `pl.ripley_plot` overlays them. A reference line at zero marks CSR: curves that rise clearly above their simulated envelope indicate genuine clustering at that distance.

To reduce CPU time on this example notebook, we subsample to 10,000 cells per cluster.

In [ ]:
rng = np.random.default_rng(42)
max_per_group = 10_000

xen = mdata['xenium']
mask = np.zeros(xen.n_obs, dtype=bool)
for g in xen.obs['leiden'].cat.categories:
    idx = np.where(xen.obs['leiden'] == g)[0]
    chosen = rng.choice(idx, min(len(idx), max_per_group), replace=False)
    mask[chosen] = True

xen_sub = xen[mask].copy()
print(f"Subsampled to {xen_sub.n_obs} cells ({max_per_group} per group max)")

ripley_curves = ab.tl.ripley_l(xen_sub, groupby='leiden', seed=42, n_jobs=4)  # Expensive test, so let's parallelize
ab.pl.ripley_plot(xen_sub).show()
xen_sub.obs['leiden'].value_counts()

All clusters show L(r) > 0 at short distances (10–50 µm): every cell type clusters at the scale of individual cells, since cells sharing a Leiden label are transcriptomically similar and tend to be physically adjacent. What the curve does at larger r is where cluster types actually differ: a curve that keeps climbing across the whole distance range indicates a spatially contiguous domain (that cell type forms one or a few large, compact regions rather than many small scattered ones); a curve that peaks and turns back down toward the reference line indicates clustering that's real but confined to a limited length scale, with cells of that type more independently scattered at longer range. Sparse or rare clusters need extra caution when reading L(r) directly: with few cells, a handful of tight local groups can produce a curve that looks dramatically "more clustered" than a common, spatially diffuse cell type without a correspondingly large biological difference — check the underlying cell count (printed below) and look at the cluster directly on the tissue (`pl.points`) before concluding one cluster is more spatially organized than another from the L(r) curve alone.

## 7. Scaling up: every gene at once

`EPCAM` was chosen based on prior biological knowledge. `tl.spatially_variable_genes` applies the same global-autocorrelation logic from section 1 across the entire panel at once, using the fast analytic p-value approximation (`n_perms=0`) rather than permutation testing gene by gene. Results are ranked by Moran's I and written to both `var` and `uns`.

In [ ]:
svg = ab.tl.spatially_variable_genes(mdata, n_top_genes=20)
svg.sort_values('score', ascending=False).head(10)

EPCAM ranks highly, meaning it is spatially associated/concentrated. Spatial autocorrelation effectively re-orders genes by their tissue-architecture specificity rather than their overall expression level.

`pl.moran_plot` provides a geometric view of the global statistic: standardized expression on the x-axis against its spatial lag on the y-axis. The slope of the best-fit line through the origin is Moran's I: a tight, positive scatter is the geometric signature of global spatial organization. Because `moran_local` for `EPCAM` was computed in section 2, the points are colored by LISA quadrant automatically.

In [ ]:
gene_rank = svg.sort_values('score', ascending=False).index.get_loc(gene)
print(f"{gene} ranks {gene_rank + 1} / {svg.shape[0]} genes by Moran's I")

ab.pl.moran_plot(mdata, gene).show()

The Moran scatter plot geometry illustrates why EPCAM ranked highly: HH cells (high EPCAM, high-EPCAM neighborhood) occupy the top-right quadrant as a tight cluster, while LL cells (the stromal majority) anchor the bottom-left. The small HL cloud represents isolated EPCAM-high cells in a low-EPCAM neighborhood: potential invasion-front cells at the tumor-stroma interface.

## Where to go next

This notebook discussed spatial organization: global Moran's I / Geary's C for whether a gene is clustered at all, local LISA / Getis-Ord for where, bivariate Moran's I for whether two signals track each other, spatial lag for de-noising, spatial silhouette for cluster-boundary quality, Ripley's L for cell-type point patterns, and a genome-wide scan for spatially variable genes without guessing in advance.

Knowing which genes are spatially organized only establishes that a pattern exists, not what causes it or which cell types are nearby when it occurs. `03_neighborhood_analysis.ipynb` covers cell-type adjacency and whether physical contact changes biology.